# Discovering Reddit communities
02807 Computational Tools for Data Science, Group 31

In [1]:
# Import necessary libraries
from pathlib import Path

import pandas as pd

import helper_functions as hp

### Define data

In [2]:
data_folder = Path("data")
posts = pd.read_csv(data_folder / "datascience_posts_2026-09.csv", parse_dates=["created_at"])
comments = pd.read_csv(data_folder / "datascience_comments_2026-09.csv", parse_dates=["created_at"])

print(posts.columns.tolist())   # posts need an "id" and an "author" column
print("subreddits:", comments["subreddit"].nunique())

df = comments[~comments["author"].isin(["[deleted]", "AutoModerator"])].dropna(subset=["author", "body"])

['id', 'subreddit', 'author', 'created_utc', 'title', 'selftext', 'score', 'num_comments', 'url', 'created_at']
subreddits: 1


### Run combined model

In [3]:
adj = hp.build_reply_graph(df, posts, min_degree=1)
levels = hp.louvain(adj)
for i, (m, q) in enumerate(levels):
    print("level", i, "communities:", len(set(m.values())), "modularity:", round(q, 3))

# size distribution of ALL level-0 communities (before dropping small ones)
print(pd.Series(levels[0][0]).value_counts().describe())

# keep the communities with at least 5 users, then name them by their top words
user_comm = hp.select_communities(levels, level=0, min_comm_size=5)
print("communities kept:", len(set(user_comm.values())), "| users covered:", len(user_comm), "of", len(adj))

labels = hp.label_words(df, user_comm, min_count=5)
names = hp.community_names(labels, n_words=3)

summary = pd.DataFrame({
    "name": pd.Series(names),
    "users": pd.Series(user_comm).value_counts(),
})
summary.index.name = "community"
print(summary.sort_values("users", ascending=False).head(10))

if comments["subreddit"].nunique() > 1:
    matches = hp.match_subreddits(df, user_comm)
    hp.report(matches, labels)
else:
    for c, row in summary.iterrows():
        print(f"Community {c} ({row['users']} users): {row['name']}")

level 0 communities: 76 modularity: 0.723
level 1 communities: 35 modularity: 0.811
level 2 communities: 28 modularity: 0.817
count    76.000000
mean      5.157895
std       5.075569
min       2.000000
25%       2.000000
50%       3.000000
75%       6.000000
max      27.000000
Name: count, dtype: float64
communities kept: 23 | users covered: 259 of 392
                                     name  users
community                                       
0                   got / job / stability     27
1                 podcast / youtube / com     21
2                      llms / using / llm     20
3          stats / statistics / questions     15
4           promotion / machine / applied     15
5               marketing / fun / product     14
6               meta / seattle / director     14
7             science / post / calculator     13
8               government / hours / tech     11
9              test / experiments / ideas     11
Community 0 (27 users): got / job / stability
Community 1

### Evaluation through null model and seed stability

In [4]:
res = hp.null_model_test(adj, n_null=20, level=0)
print(f"real Q = {res['real']:.3f} | null = {res['null_mean']:.3f} ± {res['null_std']:.3f} | z = {res['z']:.1f}")

stab = hp.seed_stability(adj, n_seeds=10, level=0)
print(stab)

real Q = 0.623 | null = 0.556 ± 0.008 | z = 8.2
{'nmi_mean': 0.9843425644774509, 'nmi_std': 0.005689014744260353, 'q_mean': 0.7208157327670035, 'q_std': 0.007949842046088305}
